In [ ]:
!pip install -q crewai crewai-tools

import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY2")

from crewai import LLM
llm = LLM(
    model = "gemini/gemini-flash-lite-latest",
    temperature = 0.2
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48

In [ ]:
job_description = """Junior Python Developer.
Must have: Python, SQL, basic Git.
Nice to have : Django or Flask, any internship.
0-2 years experience."""

resumes = {
    "anjali.txt": """Anjali Rao. B.Tech CSE 2025.
Skills: Python, SQL, Git, Flask.
Internship: 3 months backend intern at a startup, built REST APIs in Flask.
Project: Library management system with Python and MySQL.""",

    "rohit.txt": """Rohit Verma. B.Com 2024.
Skills: MS Excel, Tally, basic accounting.
Experience: 1 year as accounts assistant.
Interested in moving into tech.""",

    "meera.txt": """Meera Iyer. MCA 2025.
Skills: Python, basic SQL.
Project: Student attendance app in Python.
No internship yet. Learning Git.""",
}

for name, text in resumes.items():
  with open(name, "w") as f:
    f.write(text)


print("Saved : ",list(resumes.keys()))


Saved :  ['anjali.txt', 'rohit.txt', 'meera.txt']


In [ ]:
from crewai import Agent, Task, Crew
from crewai.flow.flow import Flow, start, listen, router, or_
import csv

class ResumeFlow(Flow):

    @start()
    def load_resume(self):
        with open(self.state["file"]) as f:
            self.state["resume"] = f.read()
        print("\n📄 Loaded:", self.state["file"])

    @listen(load_resume)
    def score_resume(self):
        answer = llm.call(
            "Score how well this resume fits the job, from 0 to 10. "
            "Reply with only the number.\n\n"
            "Job:\n" + job_description + "\n\nResume:\n" + self.state["resume"]
        )
        try:
            self.state["score"] = int(answer.strip())
        except ValueError:
            self.state["score"] = 0
        print("🔢 Score:", self.state["score"])

    @router(score_resume)
    def check_score(self):
        if self.state["score"] >= 7:
            return "shortlist"
        return "reject"

    @listen("shortlist")
    async def summarize(self):
        recruiter = Agent(
            role="Technical Recruiter",
            goal="Summarise why a candidate fits a job and what is missing",
            backstory="You are fair and specific, and only mention skills "
                      "actually written in the resume.",
            llm=llm,
            verbose=False,
        )
        task = Task(
            description="Job:\n{job}\n\nResume:\n{resume}\n\n"
                        "Write 3 strengths and 2 gaps of this candidate.",
            expected_output="3 strengths and 2 gaps.",
            agent=recruiter,
        )
        crew = Crew(agents=[recruiter], tasks=[task], verbose=False)
        result = await crew.kickoff_async(inputs={"job": job_description,
                                                  "resume": self.state["resume"]})
        self.state["decision"] = "Shortlisted"
        self.state["notes"] = result.raw

    @listen("reject")
    def draft_rejection(self):
        email = llm.call(
            "Write a short, polite rejection email (3 lines) for this candidate "
            "applying to a Junior Python Developer role.\n\nResume:\n"
            + self.state["resume"]
        )
        self.state["decision"] = "Rejected"
        self.state["notes"] = email.strip()

    @listen(or_(summarize, draft_rejection))
    def save_results(self):
        with open("results.csv", "a", newline="") as f:
            csv.writer(f).writerow([self.state["file"], self.state["score"],
                                    self.state["decision"], self.state["notes"]])
        print(self.state["decision"])

In [ ]:
import os

if os.path.exists("results.csv"):
    os.remove("results.csv")

for file in resumes.keys():
    flow = ResumeFlow()
    await flow.kickoff_async(inputs={"file": file})

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ResumeFlow                                                                                               │
│  ID: b748634b-6ad0-446b-8960-b588717c6090                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ResumeFlow                                                                                               │
│  ID: b748634b-6ad0-446b-8960-b588717c6090                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: load_resume                                                                                            │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


📄 Loaded: anjali.txt


╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: load_resume                                                                                            │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: draft_rejection                                                                                        │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Rejected


╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: save_results                                                                                           │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: save_results                                                                                           │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ResumeFlow                                                                                               │
│  ID: cbb6a49b-4dcc-4de0-b99f-5f3405724e7a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
import pandas as pd

df = pd.read_csv("results.csv", names=["file", "score", "decision", "notes"])
df

,file,score,decision,notes
0,anjali.txt,10,Shortlisted,"As a Technical Recruiter, here is my assessmen..."
1,rohit.txt,0,Rejected,"Dear Rohit,\n\nThank you for your interest in ..."
2,meera.txt,6,Rejected,"Dear Meera,\n\nThank you for your interest in ..."


In [ ]:
for i, row in df.iterrows():
    print("=" * 60)
    print(row["file"], "| Score:", row["score"], "|", row["decision"])
    print("-" * 60)
    print(row["notes"])

anjali.txt | Score: 10 | Shortlisted
------------------------------------------------------------
As a Technical Recruiter, here is my assessment of Anjali Rao for the Junior Python Developer position based strictly on her resume:

**Strengths:**
1. **Core Technical Stack Alignment:** She possesses all the "Must have" skills requested for the role, specifically Python, SQL, and Git.
2. **Framework Experience:** She meets the "Nice to have" requirement by knowing Flask, which is directly complemented by her hands-on project (Library management system with Python and MySQL).
3. **Practical Industry Experience:** She fulfills the "Nice to have" internship criterion through her 3-month backend internship at a startup, where she gained direct, relevant experience building REST APIs in Flask.

**Gaps:**
1. **Alternative Framework Knowledge:** While she knows Flask, she does not mention experience with Django (the other option listed in the nice-to-haves).
2. **Advanced Git/Tooling Depth:** T